# Protocol variants → SpatialData

`cpio.read_plate` reads one plate of a Cell Painting Gallery source into a [SpatialData](https://spatialdata.scverse.org) object.
Fields of view become Images, the CellProfiler Nuclei, Cells and Cytoplasm segmentations become Labels, the wells become Shapes where the source recorded stage coordinates, and the well- and cell-level measurements become the `wells` and `cells` Tables.

Only some fields of a well were analysed, so most images here have no labels beside them.

One well of `cpg0001-cellpainting-protocol`, fetched with the script beside this notebook:

```bash
python fetch.py cpg0001-cellpainting-protocol/source_4 2020_06_25_Stain2_Batch2_Binned BR00112197 A01
```

In [ ]:
from __future__ import annotations

import logging
import warnings
from pathlib import Path

import numpy as np
import spatialdata as sd
import spatialdata_plot  # noqa: F401
from matplotlib.colors import Normalize

import cell_painting_io as cpio

# ome-zarr logs a line per label element on read, and spatialdata-plot warns once per element it subsets out
logging.getLogger("ome_zarr").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="The table is annotating .*, which is not present")

DATA_ROOT = Path("~/data/cpg0001-cellpainting-protocol/source_4").expanduser()
BATCH = "2020_06_25_Stain2_Batch2_Binned"
PLATE = "BR00112197"
CHANNEL = "DNA"

In [ ]:
sdata = cpio.read_plate(DATA_ROOT, BATCH, PLATE, profile="normalized")
sdata

In [ ]:
fov = next(name.removesuffix("_cells") for name in sdata.labels if name.endswith("_cells"))
values = sd.get_pyramid_levels(sdata[f"{fov}_image"], n=0).sel(c=CHANNEL).to_numpy()
(
    sdata.pl.render_images(
        f"{fov}_image", channel=CHANNEL, cmap="gray", norm=Normalize(*np.percentile(values, [50, 99.9])), colorbar=False
    )
    .pl.render_labels(f"{fov}_cells", color="AreaShape_Area", table_name="cells", fill_alpha=0.55)
    .pl.show(coordinate_systems=fov, figsize=(6.5, 6), title=f"{fov}, cell masks by area")
)